### Build a Simple LLM Application with LCEL
This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting.

Wht used in it-

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [23]:
### Using GROQ API Key Model

import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
print("API key loaded:", bool(groq_api_key))

API key loaded: True


In [11]:
from langchain_groq import ChatGroq
model=ChatGroq(model="qwen/qwen3.6-27b",groq_api_key=groq_api_key)
model

ChatGroq(output_version=None, profile={}, client=<groq.resources.chat.completions.Completions object at 0x000001C4FB9AD8E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C4FB9D3F80>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [20]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

result=model.invoke(messages)

In [13]:
result

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Input text: "Hello How are you?"\n   - Source language: English\n   - Target language: French\n   - Task: Translation\n\n2.  **Identify Key Components:**\n   - "Hello" -> Greeting\n   - "How are you?" -> Inquiry about well-being\n   - Note: There\'s a missing space after "Hello" in the input ("Hello How are you?"), but that\'s a minor typo. I\'ll translate it naturally as "Hello, how are you?"\n\n3.  **Determine French Equivalents:**\n   - "Hello" -> "Bonjour" (standard, polite)\n   - "How are you?" -> "Comment allez-vous ?" (formal/plural) or "Comment vas-tu ?" (informal/singular)\n   - Since context isn\'t specified, I\'ll provide the most common/standard translation, which is informal/singular in everyday conversation: "Bonjour, comment ça va ?" or "Bonjour, comment vas-tu ?"\n   - I\'ll go with the most natural and widely used: "Bonjour, comment ça va ?" or "Bonjour, comment allez-vous ?"

In [14]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Input text: "Hello How are you?"\n   - Source language: English\n   - Target language: French\n   - Task: Translation\n\n2.  **Identify Key Components:**\n   - "Hello" -> Greeting\n   - "How are you?" -> Inquiry about well-being\n   - Note: There\'s a missing space after "Hello" in the input ("Hello How are you?"), but that\'s a minor typo. I\'ll translate it naturally as "Hello, how are you?"\n\n3.  **Determine French Equivalents:**\n   - "Hello" -> "Bonjour" (standard, polite)\n   - "How are you?" -> "Comment allez-vous ?" (formal/plural) or "Comment vas-tu ?" (informal/singular)\n   - Since context isn\'t specified, I\'ll provide the most common/standard translation, which is informal/singular in everyday conversation: "Bonjour, comment ça va ?" or "Bonjour, comment vas-tu ?"\n   - I\'ll go with the most natural and widely used: "Bonjour, comment ça va ?" or "Bonjour, comment allez-vous ?" depending on form

In [15]:
### Using LCEL-we can chain the components
chain=model|parser
chain.invoke(messages)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Source text: "Hello How are you?"\n   - Target language: French\n   - Note: There\'s a missing space/punctuation in the original ("Hello How are you?"), but it\'s clearly meant to be "Hello, How are you?" or "Hello. How are you?"\n\n2.  **Identify Key Components:**\n   - "Hello" -> French: "Bonjour"\n   - "How are you?" -> French: "Comment allez-vous ?" (formal/plural) or "Comment vas-tu ?" (informal/singular) or "Comment ça va ?" (common/neutral)\n\n3.  **Determine Appropriate Translation:**\n   - Since no context is provided, it\'s best to provide a standard, polite translation that works in most situations.\n   - "Bonjour, comment allez-vous ?" (formal)\n   - "Bonjour, comment vas-tu ?" (informal)\n   - I\'ll provide the most common/neutral option and note the formality difference if needed, but for a direct translation, "Bonjour, comment allez-vous ?" or "Bonjour, comment ça va ?" are good.\n   - Actually,

In [ ]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template=(
    "Translate the user's English text into {language}. "
    "Return only the translation. Do not include explanations, alternatives, "
    "metadata, or hidden reasoning. Preserve the original meaning and formatting."
)

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)

In [17]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [18]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [21]:
## Chaining together components with LCEL
chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello"})

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Translate the following into French:"\n   - Input to translate: "Hello"\n\n2.  **Identify Key Task:**\n   - Translate the English word "Hello" into French.\n\n3.  **Determine Translation:**\n   - "Hello" in French can be translated in several ways depending on context:\n     - Informal: "Salut"\n     - Formal/Standard: "Bonjour"\n     - Very informal/slang: "Coucou", "Hey" (less common)\n   - Since no context is provided, the most standard and widely applicable translation is "Bonjour". "Salut" is also acceptable but more informal. I\'ll provide both or the most common one with a brief note if needed, but typically just "Bonjour" is sufficient.\n\n4.  **Formulate Response:**\n   - Direct translation: "Bonjour"\n   - Keep it concise as requested.\n   - Could add context if helpful, but the prompt is straightforward.\n\n   Draft: "Bonjour" (or "Salut" for informal contexts)\n\n5.  **Final Output Gene